# Stage 12A: raw multi-scale NCC alignment benchmark

This standalone Colab notebook fixes the Stage 11C `w075_cap50` surface prior and measures how well raw horizontal/typewell GR normalized correlation ranks the true residual TVT offset. It evaluates ordinary, spatial, typewell, and pseudo-cut consistency. It does not create a submission.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess

REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
data_dir = drive_root / 'data'
artifact_dir = drive_root / 'artifacts'
assert (data_dir / 'train').is_dir(), f'Missing: {data_dir / "train"}'
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
artifact_dir.mkdir(parents=True, exist_ok=True)
subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], check=True)


## Reuse validated surface artifacts

Stage 11 and Stage 11C are reused. The fallback commands run only when the corresponding Drive artifact is absent.


In [ ]:
STAGE11_RUN_ID = 'stage11_multicut_delta_u_full_v001'
STAGE11C_RUN_ID = 'stage11c_delta_u_robust_gate_full_v001'
stage11_run = artifact_dir / STAGE11_RUN_ID
stage11c_run = artifact_dir / STAGE11C_RUN_ID
if not (stage11_run / 'gate_summary.json').is_file():
    subprocess.run([
        'uv', 'run', 'rogii-delta-u',
        '--config', 'configs/experiment/stage11_multicut_delta_u.yaml',
        '--data-dir', str(data_dir), '--artifact-dir', str(artifact_dir),
        '--run-id', STAGE11_RUN_ID,
    ], cwd=repo_dir, check=True)
if not (stage11c_run / 'gate_summary.json').is_file():
    subprocess.run([
        'uv', 'run', 'rogii-delta-u-gate',
        '--config', 'configs/experiment/stage11c_delta_u_robust_gate.yaml',
        '--stage11-run', str(stage11_run),
        '--data-dir', str(data_dir), '--artifact-dir', str(artifact_dir),
        '--run-id', STAGE11C_RUN_ID,
    ], cwd=repo_dir, check=True)
stage11c = json.loads((stage11c_run / 'gate_summary.json').read_text(encoding='utf-8'))
assert stage11c['promoted_to_stage12'] is True
assert stage11c['selected_inference_spec'] == 'w075_cap50'
print('Surface prior:', stage11c['selected_inference_parameters'])


## Run raw NCC benchmark

Use CPU/high RAM. `LIMIT_WELLS = None` is required for the decision run; 24 is only a wiring smoke test. The benchmark evaluates 61 offsets from -60 to +60 ft and may take longer than Stage 11C because it scans GR windows for all cuts and three holdout families.


In [ ]:
RUN_ID = 'stage12a_raw_ncc_benchmark_full_v001'
LIMIT_WELLS = None
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'benchmark_summary.json').is_file():
    command = [
        'uv', 'run', 'rogii-raw-ncc',
        '--config', 'configs/experiment/stage12a_raw_ncc_benchmark.yaml',
        '--stage11-run', str(stage11_run),
        '--stage11c-run', str(stage11c_run),
        '--data-dir', str(data_dir), '--artifact-dir', str(artifact_dir),
        '--run-id', RUN_ID,
    ]
    if LIMIT_WELLS is not None:
        command += ['--limit-wells', str(LIMIT_WELLS)]
    log_path = artifact_dir / f'{RUN_ID}_driver.log'
    with log_path.open('w', encoding='utf-8') as log_handle:
        process = subprocess.Popen(
            command, cwd=repo_dir, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in process.stdout:
            print(line, end='')
            log_handle.write(line); log_handle.flush()
        return_code = process.wait()
    if return_code != 0:
        tail = log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-120:]
        print('\n'.join(tail))
        raise RuntimeError(f'Stage 12A failed with exit code {return_code}. Full log: {log_path}')
else:
    print('Reusing completed run:', run_dir)


## Emission decision

Raw aligned RMSE is diagnostic: repeated geology can make raw argmin worse even when the correct state is ranked highly. Promotion to learned emission is based primarily on valid coverage, true-state rank, Top-K recall versus random, fold/holdout consistency, and oracle headroom.


In [ ]:
summary = json.loads((run_dir / 'benchmark_summary.json').read_text(encoding='utf-8'))
primary = summary['standard_primary']
{
    'promoted_to_learned_emission': summary['promoted_to_learned_emission'],
    'surface_spec': summary['surface_spec'],
    'surface_rmse': primary['surface_rmse'],
    'raw_aligned_rmse': primary['aligned_rmse'],
    'raw_aligned_delta': primary['aligned_rmse_delta'],
    'oracle_rmse': primary['oracle_rmse'],
    'oracle_delta': primary['oracle_rmse_delta'],
    'offset_in_grid_fraction': primary['offset_in_grid_fraction'],
    'emission_valid_fraction': primary['emission_valid_fraction'],
    'median_true_state_rank': primary['median_true_state_rank'],
    'top5_recall': primary['top5_recall'],
    'top10_recall': primary['top10_recall'],
    'random_top10_recall': primary['random_top10_recall'],
    'fold_top10_recall': primary['fold_top10_recall'],
    'spatial_top10_recall': summary['spatial_primary']['top10_recall'],
    'typewell_top10_recall': summary['typewell_primary']['top10_recall'],
    'gates': summary['gates'],
    'cut_report': summary['cut_report'],
    'next_step': summary['next_step'],
}


In [ ]:
import pandas as pd
pd.read_parquet(run_dir / 'variant_report.parquet').sort_values(
    ['family', 'median_true_state_rank', 'top10_recall'], ascending=[True, True, False]
).reset_index(drop=True)


## Send back

Send the complete decision dictionary and variant table. These numbers determine the Stage 12B training target, window curriculum, and whether the neural model predicts a state distribution or a pairwise margin.
